In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import os

from loguru import logger
import tqdm
import glob

In [3]:
from xgboost import DMatrix, XGBRegressor 

import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score
from sktime.performance_metrics.forecasting import (
    MeanAbsoluteScaledError, 
    MeanAbsolutePercentageError, 
    MeanSquaredError, 
    MeanAbsoluteError
)

In [4]:
import helper as hl
import models as ml
import dataset as ds

# Define Global Variables

In [5]:
SR_FREQ = '12H'
device = torch.device('cpu')

In [6]:
cli_args = dict(
    # LSTM PARAMS
    data='paloalto',  # dundee | porto | boulder | paloalto
    # DATA PARAMS
    sr_freq=SR_FREQ, min_pts=100,
    # TRAINING PARAMS
    n_rounds=40,
    # FEDERATION PARAMS
    # mu_prox = 1e-1,
    mu_prox = 0.0,
)

fedxgbllr__feat_eng_params = dict(
    time_axis='timestamp',
    X_feats=[
        'power_output_kW',
        'no_of_sessions',
        'charging_time',
        # 
        'week_sin', 'week_cos', 
        'day_sin', 'day_cos',
        'hour_sin', 'hour_cos',
        # 
        'power_curr_lag1',
        'power_curr', 
        'power_curr_logdelta',
        'power_next_step1_extrap', 
        #  
        'power_curr_std', 
        'power_curr_ema', 
        # 
        'power_curr_lag5', 'power_curr_lag48',
        'power_curr_ema_lag24', 'power_curr_ema_lag48',
        'power_curr_std_lag24', 'power_curr_std_lag48',
        #
        f'downtime_scaled',
    ],
    y_feat='power_next',
)

fedxgbllr__xgb_params = dict(
    objective="reg:quantileerror", 
    quantile_alpha=0.7, 
    n_estimators=37,
    enable_categorical=True,
)

fedxgbllr__cnn_params = dict(
    trees_per_client=fedxgbllr__xgb_params['n_estimators'],
    num_clients=8 if cli_args['data'] != 'porto' else 4,
    in_channels=1,
    conv_channels=16,
    out_channels=1,
)

# Load Dataset

In [7]:
df_tr_dev_test = pd.concat(
    {
        cluster_id: pd.read_pickle(
            cluster_id_path
        )
        # 
        for cluster_id, cluster_id_path in enumerate(
            sorted(
                glob.glob(
                    f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}_'+\
                    f'tabular_{len(fedxgbllr__feat_eng_params["X_feats"])}_xgb_inputs_'+\
                    f'{len(fedxgbllr__feat_eng_params["y_feat"])}_outputs.cluster_*.v4.pickle'
                )
            )
        )
    },
    names=['cluster_id']
).sort_values(fedxgbllr__feat_eng_params['time_axis'])

In [8]:
df_test_X = df_tr_dev_test.loc[
    df_tr_dev_test['dataset_tr1_dev2_test3'] == 3, fedxgbllr__feat_eng_params['X_feats']
].copy()

df_test_y = df_tr_dev_test.loc[
    df_tr_dev_test['dataset_tr1_dev2_test3'] == 3, [fedxgbllr__feat_eng_params['y_feat']]
].copy()

# Load Model

In [9]:
fedstrategy = 'fedavg' if cli_args['mu_prox'] == 0 else 'fedprox'

fededf_heavy_dirs = {
    f'dundee_xgb_fedavg': f'fededf_ver2025-10-07_10-46-47_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.0',
    f'dundee_xgb_fedprox': f'fededf_ver2025-10-13_20-19-28_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    f'porto_xgb_fedavg': f'fededf_ver2025-10-07_11-21-54_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.0',
    f'porto_xgb_fedprox': f'fededf_ver2025-10-13_20-20-02_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    f'boulder_xgb_fedavg': f'fededf_ver2025-10-13_07-04-36_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.0',
    f'boulder_xgb_fedprox': f'fededf_ver2025-10-13_19-49-13_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    f'paloalto_xgb_fedavg': f'fededf_ver2025-10-13_07-06-48_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.0',
    f'paloalto_xgb_fedprox': f'fededf_ver2025-10-13_19-48-35_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
}

fededf_light_dirs = {
    f'dundee_xgb_fedavg': f'fededf_ver2025-11-30_11-08-58_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.0',
    f'dundee_xgb_fedprox': f'fededf_ver2025-12-04_09-15-32_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
}

fededf_dirs = fededf_heavy_dirs

In [10]:
MODEL_XGB_TYPE_NAME = f'FedXGBllr__mu_prox_{cli_args["mu_prox"]}'
                    
save_path_best = os.path.join(
    '..', 
    'data', 
    'pth', 
    fededf_dirs[f'{cli_args["data"]}_xgb_{fedstrategy}'],
    f'fededf_{cli_args["data"]}.flwr_global.epoch{cli_args["n_rounds"]}.pth' )

print(save_path_best)

../data/pth/fededf_ver2025-10-13_07-06-48_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.0/fededf_paloalto.flwr_global.epoch40.pth


In [11]:
def load_model(path, model_config, device):
    # # Evaluate Best Model
    checkpoint = torch.load(path, map_location=device)

    fededf_model = ml.FedXGBllrCNN(
        **model_config
    )
    fededf_model.to(device)

    # Assign each NumPy array to the corresponding layer in the model
    with torch.no_grad():  # Disable gradient tracking to avoid issues during assignment
        for param, np_array in zip(fededf_model.parameters(), checkpoint['parameters']):
            # Convert NumPy array to a torch tensor with the same dtype as model parameters
            param.copy_(torch.tensor(np_array, dtype=param.dtype))

    fededf_model.eval()
    return fededf_model

# Make Predictions

In [12]:
# Load 1D-CNN
fedxgb_cnn = load_model(save_path_best, fedxgbllr__cnn_params, device)

# Load aggregated trees
agg_xgb_trees = np.load(
    os.path.join(
        '../data/pth',
        fededf_dirs[f'{cli_args["data"]}_xgb_{fedstrategy}'],
        f'fededf_{cli_args["data"]}.flwr_global.epoch0.xgb_trees.npy'
    ),
    allow_pickle=True
)

In [13]:
fedxgb_cnn

FedXGBllrCNN(
  (conv1d): Conv1d(1, 16, kernel_size=(37,), stride=(37,))
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [14]:
# Create model input
def xgb_tree_predictions(xgb_model, X, **kwargs):
    booster = xgb_model.get_booster()
    n_estimators = len(booster.get_dump())

    X_dm = DMatrix(X, **kwargs)

    tree_preds = []
    for tree_ix in range(n_estimators):
        tree_preds.append(
            booster.predict(X_dm, iteration_range=(tree_ix, tree_ix+1), output_margin=True)
        )

    return np.vstack(tree_preds).T  # <n_samples, n_estimators>


def fedxgbllr_cnn_create_dataloader(aggregated_trees, dataset, **kwargs):
    X = []
    
    for client_id_tree, _ in aggregated_trees:
        X.append(
            xgb_tree_predictions(
                client_id_tree, dataset['X'], **kwargs
            )
        )

    X, y = torch.from_numpy(
        np.expand_dims(
            np.concatenate(X, axis=1), 
            axis=-2
        )
    ), torch.from_numpy(
        np.expand_dims(
            np.array(dataset['y']),
            axis=-1
        )
    )

    return TensorDataset(X, y)


df_test_Xy = hl.fedxgbllr_cnn_create_dataloader(
    agg_xgb_trees, dataset={'X':df_test_X, 'y':df_test_y}, logger=logger, **{'enable_categorical':True}
)

2026-02-14 15:24:45.675 | INFO     | helper:fedxgbllr_cnn_create_dataloader:601 - [fedxgbllr_cnn_create_dataloader] Fetching the individual predictions of tree #0...
2026-02-14 15:24:45.687 | INFO     | helper:fedxgbllr_cnn_create_dataloader:601 - [fedxgbllr_cnn_create_dataloader] Fetching the individual predictions of tree #1...
2026-02-14 15:24:45.698 | INFO     | helper:fedxgbllr_cnn_create_dataloader:601 - [fedxgbllr_cnn_create_dataloader] Fetching the individual predictions of tree #2...
2026-02-14 15:24:45.710 | INFO     | helper:fedxgbllr_cnn_create_dataloader:601 - [fedxgbllr_cnn_create_dataloader] Fetching the individual predictions of tree #3...
2026-02-14 15:24:45.722 | INFO     | helper:fedxgbllr_cnn_create_dataloader:601 - [fedxgbllr_cnn_create_dataloader] Fetching the individual predictions of tree #4...
2026-02-14 15:24:45.733 | INFO     | helper:fedxgbllr_cnn_create_dataloader:601 - [fedxgbllr_cnn_create_dataloader] Fetching the individual predictions of tree #5...
2026

In [15]:
# Get results
y_pred_cnn = fedxgb_cnn(df_test_Xy.tensors[0]).clip(min=0)

# Merge with actual data
df_test_y.loc[:, MODEL_XGB_TYPE_NAME] = y_pred_cnn.detach().numpy()

In [16]:
# Evaluate results
edf_results = df_tr_dev_test[
    ['oid', 'timestamp']
].join(
    df_test_y
).dropna().set_index(
    ['oid', 'timestamp']
)

# Calculate metrics

In [17]:
available_oids = list(
    set(
        df_tr_dev_test.loc[
            df_tr_dev_test.dataset_tr1_dev2_test3 == 3,
            'oid'
        ]
    ).intersection(
        set(
            df_tr_dev_test.loc[
                df_tr_dev_test.dataset_tr1_dev2_test3 == 1,
                'oid'
            ]
        )
    )
)

y_train = df_tr_dev_test.loc[
    (
        df_tr_dev_test.dataset_tr1_dev2_test3 == 1
    ) 
    & (
        df_tr_dev_test.oid.isin(available_oids)
    )
].reset_index(level=0, drop=True).set_index(['oid', 'timestamp'], append=True)[
    ['power_curr', 'power_output_kW']
].sort_index()

# Get the energy demand from each EVSE of the train set
y_train = y_train.power_curr * y_train.power_output_kW.astype(float) * (pd.Timedelta(cli_args['sr_freq'].lower()).total_seconds() / 3600)

# Get the indices of each value per object id 
oid_indices = y_train.sort_index().groupby('oid', observed=False).groups

In [18]:
model_results_metrics = hl.evaluate_predictions(
    edf_results.dropna(),
    y_true_name='power_next',
    y_pred_names=[MODEL_XGB_TYPE_NAME],
    eval_funs=[
        ('MASE_pct', MeanAbsoluteScaledError(sp=24), {'y_train':y_train, 'oid_indices':oid_indices}),
        ('SMAPE_pct', MeanAbsolutePercentageError(symmetric=True), {}),
        ('MAAPE_rads', hl.mean_arctangent_absolute_percentage_error, {}),
        ('WAPE_pct', hl.wape, {}),
        ('RMSE_kW', MeanSquaredError(square_root=True), {}),
        ('MAE_kW', MeanAbsoluteError(), {}),
        ('R2', r2_score, {}),
    ]
)

In [19]:
model_results_metrics.groupby(level=0, sort=False).describe().T.loc[
    pd.IndexSlice[:, ['mean', '25%', '50%', '75%']], :
]

0_MASE_pct  1_SMAPE_pct  2_MAAPE_rads  \
FedXGBllr__mu_prox_0.0 mean    1.915162     1.397855      1.186998   
                       25%     1.278024     1.180222      1.061992   
                       50%     1.605441     1.411098      1.210835   
                       75%     1.898757     1.562595      1.302938   

                             3_WAPE_pct  4_RMSE_kW   5_MAE_kW       6_R2  
FedXGBllr__mu_prox_0.0 mean   16.117689  20.702834  19.389512 -26.339242  
                       25%     1.726164  19.336816  17.448536  -5.755195  
                       50%     2.529448  20.687275  19.250444  -2.394822  
                       75%     3.906047  21.717894  20.720422  -1.419655

In [20]:
edf_results.dropna().to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__{MODEL_XGB_TYPE_NAME}__model.v4.pickle')
model_results_metrics.to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__{MODEL_XGB_TYPE_NAME}__model.metrics.v4.pickle')